# BERTopic v3 최종 버전
- 도메인별 커스텀 불용어 (CV, NLP, 로봇, 에너지...)
- TF-IDF 가중치 적용 (단어 중요도 반영)
- 토픽 병합 기능 (유사 토픽 통합)
- 토픽 자동 레이블링 (읽기 쉬운 이름)
- 최적화된 파라미터

## 패키지 설치

In [4]:
import sys
import subprocess

def install_packages():
    packages = [
        'bertopic',
        'sentence-transformers',
        'umap-learn',
        'hdbscan',
        'pyarrow'
    ]

    for package in packages:
        try:
            __import__(package.replace('-', '_'))
        except ImportError:
            print(f"{package} 설치")
            subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
    print("패키지 설치\n")

# Colab이면 패키지 자동으로 설치
try:
    import google.colab
    IN_COLAB = True
    install_packages()
except:
    IN_COLAB = False

umap-learn 설치
패키지 설치



## 임포트


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
warnings.filterwarnings('ignore')

# BERTopic 관련
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import TfidfVectorizer  # TF-IDF 사용!

# 한글 폰트 설정
import platform
system = platform.system()

if IN_COLAB:
    subprocess.run(['apt-get', '-qq', '-y', 'install', 'fonts-nanum'],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    import matplotlib.font_manager as fm
    fontpath = '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf'
    font = fm.FontProperties(fname=fontpath, size=10)
    plt.rc('font', family='NanumBarunGothic')
elif system == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
elif system == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:
    plt.rcParams['font.family'] = 'DejaVu Sans'

plt.rcParams['axes.unicode_minus'] = False

## 1. AI 분류 패턴 정의

In [6]:
TRADITIONAL_ML_PATTERNS = [
    r'\bml\b', r'machine\s+learning', r'supervised\s+learning', r'unsupervised\s+learning',
    r'\bsvm\b', r'support\s+vector', r'random\s+forest', r'decision\s+tree',
    r'k-?nearest', r'\bknn\b', r'naive\s+bayes', r'logistic\s+regression',
    r'gradient\s+boosting', r'\bxgboost\b', r'\blightgbm\b',
    r'머신\s?러닝', r'기계\s?학습', r'지도\s?학습', r'비지도\s?학습',
]

DEEP_LEARNING_PATTERNS = [
    r'deep\s+learning', r'neural\s+network', r'\bcnn\b', r'\brnn\b', r'\blstm\b',
    r'\bgru\b', r'transformer', r'attention', r'\bbert\b', r'\bgpt\b',
    r'computer\s+vision', r'object\s+detection', r'\bgan\b', r'autoencoder',
    r'resnet', r'yolo', r'vgg', r'alexnet', r'inception',
    r'딥\s?러닝', r'심층\s?학습', r'신경망', r'합성곱', r'순환\s?신경망',
    r'트랜스포머', r'어텐션', r'객체\s?탐지', r'이미지\s?분류',
]

# 논문을 전통ML 또는 딥러닝으로 분류
def classify_paper(row):

    kywd = str(row.get('KYWD', '')) if pd.notna(row.get('KYWD')) else ''
    title = str(row.get('NODE_TTLE_EN', '')) if pd.notna(row.get('NODE_TTLE_EN')) else ''
    abstract = str(row.get('ABST_EN', '')) if pd.notna(row.get('ABST_EN')) else ''

    text = (kywd + ' ' + title + ' ' + abstract).lower()

    for pattern in DEEP_LEARNING_PATTERNS:
        if re.search(pattern, text, re.IGNORECASE):
            return '딥러닝'

    for pattern in TRADITIONAL_ML_PATTERNS:
        if re.search(pattern, text, re.IGNORECASE):
            return '전통ML'

    return None

## 2. 도메인별 불용어 정의

In [7]:
# 기본 불용어
KOREAN_STOPWORDS = [
    '것', '수', '등', '및', '이', '그', '저', '개', '명', '번', '위',
    '때', '년', '월', '일', '시', '분', '초', '곳', '데', '점',
    '나', '너', '우리', '저희', '당신', '여러분', '누구', '무엇', '어디',
    '것', '거', '수', '때', '데', '바', '줄', '적', '번', '차', '대로',
    '연구', '분석', '결과', '방법', '제안', '제시', '개발', '설계',
    '적용', '활용', '이용', '사용', '구현', '시스템', '기술', '방식',
    '과정', '단계', '요소', '특징', '문제', '해결', '성능', '효과',
    '효율', '정확', '정확도', '비교', '평가', '실험', '검증', '측정',
    '향상', '개선', '최적', '최적화', '가능', '필요', '중요', '기반',
    '통해', '위해', '대해', '대한', '관련', '따른', '위한', '통한',
    '각', '각각', '모든', '여러', '다양', '주요', '전체', '일반',
    '기존', '새로운', '다른', '같은', '이러한', '그러한',
    '딥러닝', '심층', '학습', '신경망', '합성곱', '순환',
    '머신러닝', '기계학습', '지도학습', '비지도학습', '강화학습',
    '인공지능', 'ai', '트랜스포머', '어텐션',
    '데이터', '훈련', '테스트', '검증', '샘플', '입력', '출력',
]

ENGLISH_STOPWORDS = [
    'the', 'of', 'and', 'to', 'in', 'is', 'it', 'for', 'as', 'was', 'with',
    'be', 'by', 'on', 'at', 'from', 'or', 'an', 'are', 'this', 'that', 'which',
    'their', 'we', 'our', 'these', 'those', 'than', 'into', 'through', 'during',
    'before', 'after', 'above', 'below', 'between', 'under', 'again', 'further',
    'also', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such',
    'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'then', 'too', 'very',
    'can', 'will', 'just', 'should', 'now',
    'deep', 'learning', 'machine', 'neural', 'network', 'networks',
    'cnn', 'rnn', 'lstm', 'gru', 'transformer', 'transformers',
    'ml', 'dl', 'ai', 'artificial', 'intelligence',
    'data', 'model', 'models', 'modeling', 'training', 'train', 'trained',
    'test', 'testing', 'validation', 'dataset', 'datasets',
    'input', 'output', 'feature', 'features', 'parameter', 'parameters',
    'layer', 'layers', 'epoch', 'epochs', 'batch', 'batches',
    'long', 'short', 'large', 'small', 'big', 'term', 'terms',
    'pre', 'post', 'multi', 'single', 'double',
    'based', 'using', 'used', 'use', 'uses',
    'make', 'makes', 'making', 'made',
    'show', 'shows', 'shown', 'showing',
    'propose', 'proposed', 'proposing',
    'present', 'presented', 'presenting',
    'provide', 'provides', 'provided', 'providing',
    'achieve', 'achieved', 'achieving',
    'improve', 'improved', 'improving', 'improvement', 'improvements',
    'demonstrate', 'demonstrated', 'demonstrating',
    'obtain', 'obtained', 'obtaining',
    'evaluate', 'evaluated', 'evaluating', 'evaluation',
    'method', 'methods', 'methodology',
    'study', 'studies', 'research',
    'analysis', 'analyze', 'analyzed',
    'approach', 'approaches',
    'technique', 'techniques',
    'algorithm', 'algorithms',
    'system', 'systems',
    'application', 'applications', 'applied',
    'experiment', 'experiments', 'experimental',
    'paper', 'work', 'works',
    'performance', 'result', 'results',
    'accuracy', 'accurate', 'precisely', 'precision',
    'efficient', 'efficiency', 'effectively', 'effective',
    'better', 'best', 'good', 'well',
    'high', 'higher', 'highest',
    'low', 'lower', 'lowest',
    'fast', 'faster', 'fastest',
    'robust', 'robustness',
    'compare', 'compared', 'comparison', 'comparisons',
    'different', 'difference', 'differences',
    'various', 'variety',
    'several', 'multiple',
    'however', 'therefore', 'thus', 'hence',
    'consider', 'considered', 'considering',
    'given', 'since', 'while', 'although',
    'number', 'numbers',
    'one', 'two', 'three', 'four', 'five',
    'first', 'second', 'third',
    'may', 'might', 'could', 'would', 'should',
    'way', 'ways', 'manner',
    'need', 'needs', 'needed', 'requiring', 'required',
    'important', 'significance', 'significant',
    'main', 'major', 'key', 'primary',
    'general', 'specific', 'particular',
]

# 도메인별 불용어
CV_STOPWORDS = [
    'image', 'images', 'pixel', 'pixels', 'frame', 'frames',
    'visual', 'vision', 'picture', 'photo',
]

NLP_STOPWORDS = [
    'text', 'texts', 'word', 'words', 'sentence', 'sentences',
    'token', 'tokens', 'vocabulary', 'corpus',
]

ROBOTICS_STOPWORDS = [
    'robot', 'robots', 'robotic', 'robotics',
    'control', 'controller', 'controlling',
]

ENERGY_STOPWORDS = [
    'energy', 'power', 'electricity', 'electric',
]

# 통합 불용어
ALL_STOPWORDS = (KOREAN_STOPWORDS + ENGLISH_STOPWORDS +
                 CV_STOPWORDS + NLP_STOPWORDS +
                 ROBOTICS_STOPWORDS + ENERGY_STOPWORDS)

print(f"통계:")
print(f"   - 한국어: {len(KOREAN_STOPWORDS)}개")
print(f"   - 영어: {len(ENGLISH_STOPWORDS)}개")
print(f"   - CV: {len(CV_STOPWORDS)}개")
print(f"   - NLP: {len(NLP_STOPWORDS)}개")
print(f"   - Robotics: {len(ROBOTICS_STOPWORDS)}개")
print(f"   - Energy: {len(ENERGY_STOPWORDS)}개")
print(f"   - 전체: {len(set(ALL_STOPWORDS))}개 (중복 제거)\n")

통계:
   - 한국어: 125개
   - 영어: 264개
   - CV: 10개
   - NLP: 10개
   - Robotics: 7개
   - Energy: 4개
   - 전체: 412개 (중복 제거)



## 3. 데이터 로드 및 분류

In [8]:
def load_and_classify_papers(parquet_file):
    df = pd.read_parquet(parquet_file)
    print(f"{len(df):,}개 논문 로드")

    df['year'] = df['PBSH'].str[:4]
    df['ai_type'] = df.apply(classify_paper, axis=1)

    df_classified = df[df['ai_type'].notna()].copy()
    print(f"   - 전체: {len(df):,}편")
    print(f"   - 분류됨: {len(df_classified):,}편 ({len(df_classified)/len(df)*100:.1f}%)")
    print(f"   - 딥러닝: {(df_classified['ai_type'] == '딥러닝').sum():,}편")
    print(f"   - 전통ML: {(df_classified['ai_type'] == '전통ML').sum():,}편")

    papers_by_type_period = {
        '전통ML': {'2021-2022': [], '2023-2025': []},
        '딥러닝': {'2021-2022': [], '2023-2025': []}
    }

    for ai_type in ['전통ML', '딥러닝']:
        for period in ['2021-2022', '2023-2025']:
            if period == '2021-2022':
                mask = (df_classified['ai_type'] == ai_type) & (df_classified['year'].isin(['2021', '2022']))
            else:
                mask = (df_classified['ai_type'] == ai_type) & (df_classified['year'].isin(['2023', '2024', '2025']))

            filtered = df_classified[mask]

            for _, row in filtered.iterrows():
                tokens = row['merged_tokens_dedup']

                if isinstance(tokens, np.ndarray):
                    tokens = tokens.tolist()

                filtered_tokens = []
                for token in tokens:
                    token_str = str(token).strip()
                    if len(token_str) < 2:
                        continue
                    if token_str.isdigit():
                        continue
                    if not any(c.isalnum() for c in token_str):
                        continue
                    filtered_tokens.append(token_str)

                text = ' '.join(filtered_tokens)

                if len(filtered_tokens) >= 10:
                    papers_by_type_period[ai_type][period].append({
                        'text': text,
                        'tokens': filtered_tokens,
                        'year': row['year'],
                        'title': row.get('NODE_TTLE_EN', ''),
                        'keywords': row.get('KYWD', '')
                    })

    print("-" * 70)
    for ai_type in ['전통ML', '딥러닝']:
        before = len(papers_by_type_period[ai_type]['2021-2022'])
        after = len(papers_by_type_period[ai_type]['2023-2025'])
        print(f"\n{ai_type}:")
        print(f"2021-2022: {before:,}편")
        print(f"2023-2025: {after:,}편")
        print(f"총: {before + after:,}편")
    print("\n")

    return papers_by_type_period

## 4. BERTopic 모델 생성 (TF-IDF 적용)

In [9]:
def create_bertopic_model(ai_type='딥러닝', n_topics=None):

    # 임베딩: Multilingual
    embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

    # UMAP
    umap_model = UMAP(
        n_neighbors=15,
        n_components=5,
        min_dist=0.0,
        metric='cosine',
        random_state=42
    )

    if ai_type == '전통ML':
        min_cluster = 10
        min_samples = 3
        print(f"🔧 {ai_type}용 파라미터: min_cluster={min_cluster}, min_samples={min_samples}")
    else:
        min_cluster = 15
        min_samples = 5
        print(f"🔧 {ai_type}용 파라미터: min_cluster={min_cluster}, min_samples={min_samples}")

    hdbscan_model = HDBSCAN(
        min_cluster_size=min_cluster,
        min_samples=min_samples,
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True
    )

    # TF-IDF Vectorizer
    vectorizer_model = TfidfVectorizer(
        ngram_range=(1, 2),
        stop_words=ALL_STOPWORDS,
        min_df=2,
        max_df=0.85,  # 더 엄격하게
        max_features=None,
        lowercase=False,
        sublinear_tf=True,  # TF를 로그 스케일로
        norm='l2'  # L2 정규화
    )

    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        nr_topics=n_topics,
        top_n_words=15,
        verbose=False,
        calculate_probabilities=True
    )

    return topic_model

## 5. BERTopic 분석 실행 + 토픽 병합

In [10]:
def analyze_with_bertopic(papers_by_type_period, n_topics='auto', auto_reduce=True):

    results = {}

    for ai_type in ['전통ML', '딥러닝']:
        results[ai_type] = {}

        for period in ['2021-2022', '2023-2025']:
            papers = papers_by_type_period[ai_type][period]

            min_papers = 30 if ai_type == '딥러닝' else 20
            if len(papers) < min_papers:
                print(f"{ai_type} ({period}): 논문 수 부족 ({len(papers)}편), skip")
                continue

            print(f"\n{'='*70}")
            print(f"{ai_type} ({period}) BERTopic 분석")
            print(f"논문 수: {len(papers):,}편")
            print('-'*70 + '\n')

            documents = [p['text'] for p in papers]

            if n_topics == 'auto':
                num_topics = None
            else:
                num_topics = min(n_topics, len(papers) // 20)

            topic_model = create_bertopic_model(
                ai_type=ai_type,
                n_topics=num_topics
            )

            topics, probs = topic_model.fit_transform(documents)

            # 토픽 병합
            if auto_reduce:
                initial_topics = len(set(topics)) - 1  # -1 제외

                # 목표 토픽 수 계산
                target_topics = max(10, int(initial_topics * 0.7))

                try:
                    topic_model.reduce_topics(documents, nr_topics=target_topics)
                    topics = topic_model.topics_
                    final_topics = len(set(topics)) - 1
                    print(f"병합 성공: {initial_topics} → {final_topics}")
                except Exception as e:
                    print(f"원본 유지: {str(e)[:50]}")

            results[ai_type][period] = {
                'model': topic_model,
                'topics': topics,
                'probs': probs,
                'documents': documents,
                'papers': papers
            }

            topic_info = topic_model.get_topic_info()
            n_topics_found = len(topic_info[topic_info['Topic'] != -1])
            outliers = (topic_info[topic_info['Topic'] == -1]['Count'].values[0]
                       if -1 in topic_info['Topic'].values else 0)

            print(f"최종 토픽 수: {n_topics_found}개")
            print(f"아웃라이어: {outliers}개 ({outliers/len(papers)*100:.1f}%)")

            print(f"\n상위 5개 토픽:")
            for idx, row in topic_info[topic_info['Topic'] != -1].head(5).iterrows():
                topic_id = row['Topic']
                topic_words = topic_model.get_topic(topic_id)[:5]
                words = [word for word, score in topic_words]
                count = row['Count']
                print(f"   Topic {topic_id} ({count}편): {', '.join(words)}")

    return results


## 6. 토픽 자동 레이블링

In [11]:
def generate_topic_labels(results):

    for ai_type in ['전통ML', '딥러닝']:
        for period in ['2021-2022', '2023-2025']:
            if period not in results[ai_type]:
                continue

            model = results[ai_type][period]['model']
            topic_info = model.get_topic_info()

            labels = {}

            for _, row in topic_info[topic_info['Topic'] != -1].iterrows():
                topic_id = row['Topic']
                topic_words = model.get_topic(topic_id)[:3]  # 상위 3개
                words = [word for word, score in topic_words]

                # 레이블 생성 (상위 2-3개 단어 조합)
                if len(words) >= 2:
                    label = f"{words[0].title()} & {words[1].title()}"
                else:
                    label = words[0].title()

                labels[topic_id] = label

            results[ai_type][period]['labels'] = labels

            print(f"\n{ai_type} ({period}):")
            for topic_id, label in list(labels.items())[:5]:
                count = topic_info[topic_info['Topic'] == topic_id]['Count'].values[0]
                print(f"   Topic {topic_id}: {label} ({count}편)")

    return results

## 7. 시각화

In [12]:
def visualize_topics(results, output_dir='bertopic_results_v3'):

    import os
    os.makedirs(output_dir, exist_ok=True)

    for ai_type in ['전통ML', '딥러닝']:
        for period in ['2021-2022', '2023-2025']:
            if period not in results[ai_type]:
                continue

            model = results[ai_type][period]['model']

            print(f"\n{ai_type} ({period}):")

            try:
                fig = model.visualize_barchart(top_n_topics=8, n_words=10)
                filename = f"{output_dir}/{ai_type}_{period}_barchart.html".replace(' ', '_')
                fig.write_html(filename)
                print(f"막대 그래프: {filename}")
            except Exception as e:
                print(f"막대 실패: {str(e)[:50]}")

            try:
                fig = model.visualize_topics()
                filename = f"{output_dir}/{ai_type}_{period}_topics.html".replace(' ', '_')
                fig.write_html(filename)
                print(f"토픽 맵: {filename}")
            except Exception as e:
                print(f"토픽 실패: {str(e)[:50]}")

            try:
                fig = model.visualize_hierarchy()
                filename = f"{output_dir}/{ai_type}_{period}_hierarchy.html".replace(' ', '_')
                fig.write_html(filename)
                print(f"계층 구조: {filename}")
            except Exception as e:
                print(f"계층 실패: {str(e)[:50]}")


## 8. 토픽 비교

In [13]:

def compare_topics(results):

    for ai_type in ['전통ML', '딥러닝']:
        if '2021-2022' not in results[ai_type] or '2023-2025' not in results[ai_type]:
            print(f"\n{ai_type}: 비교 데이터 부족")
            continue

        print(f"{ai_type} 분야")
        print('━'*70)

        model_before = results[ai_type]['2021-2022']['model']
        model_after = results[ai_type]['2023-2025']['model']
        labels_before = results[ai_type]['2021-2022'].get('labels', {})
        labels_after = results[ai_type]['2023-2025'].get('labels', {})

        print(f"\n2021-2022 주요 토픽:")
        topics_before = model_before.get_topic_info()
        for idx, row in topics_before[topics_before['Topic'] != -1].head(5).iterrows():
            topic_id = row['Topic']
            topic_words = model_before.get_topic(topic_id)[:5]
            words = [word for word, score in topic_words]
            count = row['Count']
            label = labels_before.get(topic_id, '')
            print(f"   Topic {topic_id} - {label}")
            print(f"      ({count}편): {', '.join(words)}")

        print(f"\n2023-2025 주요 토픽:")
        topics_after = model_after.get_topic_info()
        for idx, row in topics_after[topics_after['Topic'] != -1].head(5).iterrows():
            topic_id = row['Topic']
            topic_words = model_after.get_topic(topic_id)[:5]
            words = [word for word, score in topic_words]
            count = row['Count']
            label = labels_after.get(topic_id, '')
            print(f"   Topic {topic_id} - {label}")
            print(f"      ({count}편): {', '.join(words)}")


## 9. 엑셀 저장 (레이블 포함)

In [15]:
def save_results(results, output_file='bertopic_analysis_v3.xlsx'):

    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:

        summary_data = []
        for ai_type in ['전통ML', '딥러닝']:
            for period in ['2021-2022', '2023-2025']:
                if period in results[ai_type]:
                    model = results[ai_type][period]['model']
                    topic_info = model.get_topic_info()
                    n_topics = len(topic_info[topic_info['Topic'] != -1])
                    n_docs = len(results[ai_type][period]['documents'])
                    outliers = (topic_info[topic_info['Topic'] == -1]['Count'].values[0]
                               if -1 in topic_info['Topic'].values else 0)

                    summary_data.append({
                        'AI유형': ai_type,
                        '기간': period,
                        '논문수': n_docs,
                        '토픽수': n_topics,
                        '아웃라이어': outliers,
                        '아웃라이어비율(%)': f"{outliers/n_docs*100:.1f}"
                    })

        df_summary = pd.DataFrame(summary_data)
        df_summary.to_excel(writer, sheet_name='요약', index=False)

        for ai_type in ['전통ML', '딥러닝']:
            for period in ['2021-2022', '2023-2025']:
                if period not in results[ai_type]:
                    continue

                model = results[ai_type][period]['model']
                topic_info = model.get_topic_info()
                topic_info = topic_info[topic_info['Topic'] != -1].copy()
                labels = results[ai_type][period].get('labels', {})

                # 레이블 추가
                topic_info['Label'] = topic_info['Topic'].map(labels)

                topic_words_list = []
                for topic_id in topic_info['Topic']:
                    topic_words = model.get_topic(topic_id)[:10]
                    words = ', '.join([f"{word}({score:.3f})" for word, score in topic_words])
                    topic_words_list.append(words)

                topic_info['Top_Words'] = topic_words_list

                # 컬럼 순서 조정
                cols = ['Topic', 'Label', 'Count', 'Top_Words'] + [c for c in topic_info.columns if c not in ['Topic', 'Label', 'Count', 'Top_Words']]
                topic_info = topic_info[cols]

                sheet_name = f"{ai_type}_{period}".replace('/', '-')[:31]
                topic_info.to_excel(writer, sheet_name=sheet_name, index=False)

# 메인

In [16]:
PARQUET_FILE = 'df_tokens.parquet'
N_TOPICS = 'auto'
AUTO_REDUCE = True
OUTPUT_DIR = 'bertopic_results_v3'

## 데이터 로드 및 분류

In [17]:
papers_by_type_period = load_and_classify_papers(PARQUET_FILE)
print({k: len(v) for k, v in papers_by_type_period.items()})

60,707개 논문 로드
   - 전체: 60,707편
   - 분류됨: 9,492편 (15.6%)
   - 딥러닝: 7,830편
   - 전통ML: 1,662편
----------------------------------------------------------------------

전통ML:
2021-2022: 661편
2023-2025: 995편
총: 1,656편

딥러닝:
2021-2022: 2,910편
2023-2025: 4,904편
총: 7,814편


{'전통ML': 2, '딥러닝': 2}


## BERTopic 분석 (v3)

In [18]:
results = analyze_with_bertopic(
    papers_by_type_period,
    n_topics=N_TOPICS,
    auto_reduce=AUTO_REDUCE
)


전통ML (2021-2022) BERTopic 분석
논문 수: 661편
----------------------------------------------------------------------



modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🔧 전통ML용 파라미터: min_cluster=10, min_samples=3
병합 성공: 20 → 13
최종 토픽 수: 13개
아웃라이어: 172개 (26.0%)

상위 5개 토픽:
   Topic 0 (77편): solar, 태양광, outlier, ann, renewable
   Topic 1 (76편): 신호, 측위, cable, tower, bit
   Topic 2 (68편): 금융, innovation, 프로젝트, cache, 자가
   Topic 3 (53편): subjective, spss, oral, 노인, 거주
   Topic 4 (46편): 공격, intrusion, malware, intrusion detection, attack

전통ML (2023-2025) BERTopic 분석
논문 수: 995편
----------------------------------------------------------------------

🔧 전통ML용 파라미터: min_cluster=10, min_samples=3
병합 성공: 24 → 15
최종 토픽 수: 15개
아웃라이어: 217개 (21.8%)

상위 5개 토픽:
   Topic 0 (351편): log, diagnosis, contrastive, augmentation, 파일
   Topic 1 (96편): nutrition, 국민건강, 영양조사, 국민건강 영양조사, nutrition examination
   Topic 2 (43편): unity, agents, 로봇, 에이전트, twin
   Topic 3 (42편): traffic accident, 교통사고, 교통안전, aviation, 운전자
   Topic 4 (41편): 활성, 기능성, 함량, ce, protein

딥러닝 (2021-2022) BERTopic 분석
논문 수: 2,910편
----------------------------------------------------------------------

🔧 딥러닝용 파

## 자동 토픽 라벨 생성 (v3)

In [19]:
results = generate_topic_labels(results)


전통ML (2021-2022):
   Topic 0: Solar & 태양광 (77편)
   Topic 1: 신호 & 측위 (76편)
   Topic 2: 금융 & Innovation (68편)
   Topic 3: Subjective & Spss (53편)
   Topic 4: 공격 & Intrusion (46편)

전통ML (2023-2025):
   Topic 0: Log & Diagnosis (351편)
   Topic 1: Nutrition & 국민건강 (96편)
   Topic 2: Unity & Agents (43편)
   Topic 3: Traffic Accident & 교통사고 (42편)
   Topic 4: 활성 & 기능성 (41편)

딥러닝 (2021-2022):
   Topic 0: Super Resolution & Accelerator (479편)
   Topic 1: Lane & Its (126편)
   Topic 2: 수소 & Organic (118편)
   Topic 3: 지식 그래프 & Corpu (103편)
   Topic 4: 발전량 & Solar (100편)

딥러닝 (2023-2025):
   Topic 0: Lesion & 분자 (276편)
   Topic 1: Offloading & 워크로드 (215편)
   Topic 2: Sign Language & Language Translation (177편)
   Topic 3: Encryption & 암호화 (144편)
   Topic 4: Synthetic Aperture & Aperture Radar (128편)


## 토픽 비교

In [20]:
compare_topics(results)


전통ML 분야
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

2021-2022 주요 토픽:
   Topic 0 - Solar & 태양광
      (77편): solar, 태양광, outlier, ann, renewable
   Topic 1 - 신호 & 측위
      (76편): 신호, 측위, cable, tower, bit
   Topic 2 - 금융 & Innovation
      (68편): 금융, innovation, 프로젝트, cache, 자가
   Topic 3 - Subjective & Spss
      (53편): subjective, spss, oral, 노인, 거주
   Topic 4 - 공격 & Intrusion
      (46편): 공격, intrusion, malware, intrusion detection, attack

2023-2025 주요 토픽:
   Topic 0 - Log & Diagnosis
      (351편): log, diagnosis, contrastive, augmentation, 파일
   Topic 1 - Nutrition & 국민건강
      (96편): nutrition, 국민건강, 영양조사, 국민건강 영양조사, nutrition examination
   Topic 2 - Unity & Agents
      (43편): unity, agents, 로봇, 에이전트, twin
   Topic 3 - Traffic Accident & 교통사고
      (42편): traffic accident, 교통사고, 교통안전, aviation, 운전자
   Topic 4 - 활성 & 기능성
      (41편): 활성, 기능성, 함량, ce, protein
딥러닝 분야
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

2021-2022 주요 토픽:


## 시각화 파일 생성

In [21]:
visualize_topics(results, output_dir=OUTPUT_DIR)
print(f"📁 {OUTPUT_DIR} 폴더 안에 HTML 파일 생성")


전통ML (2021-2022):
막대 그래프: bertopic_results_v3/전통ML_2021-2022_barchart.html
토픽 맵: bertopic_results_v3/전통ML_2021-2022_topics.html
계층 구조: bertopic_results_v3/전통ML_2021-2022_hierarchy.html

전통ML (2023-2025):
막대 그래프: bertopic_results_v3/전통ML_2023-2025_barchart.html
토픽 맵: bertopic_results_v3/전통ML_2023-2025_topics.html
계층 구조: bertopic_results_v3/전통ML_2023-2025_hierarchy.html

딥러닝 (2021-2022):
막대 그래프: bertopic_results_v3/딥러닝_2021-2022_barchart.html
토픽 맵: bertopic_results_v3/딥러닝_2021-2022_topics.html
계층 구조: bertopic_results_v3/딥러닝_2021-2022_hierarchy.html

딥러닝 (2023-2025):
막대 그래프: bertopic_results_v3/딥러닝_2023-2025_barchart.html
토픽 맵: bertopic_results_v3/딥러닝_2023-2025_topics.html
계층 구조: bertopic_results_v3/딥러닝_2023-2025_hierarchy.html
📁 bertopic_results_v3 폴더 안에 HTML 파일 생성


## 결과 저장

In [22]:
save_results(results)

print("완료: bertopic_analysis.xlsx")


완료: bertopic_analysis.xlsx


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [23]:
import os
import shutil
from google.colab import drive

# Google 드라이브 마운트
drive.mount('/content/drive')

# 원본 경로
source_path = '/content/bertopic_results_v3'

# 목적지 경로
destination_path = '/content/drive/MyDrive/datathon'

# 목적지 경로가 없으면 생성
if not os.path.exists(destination_path):
    os.makedirs(destination_path)

# '/content' 경로에 있는 모든 파일과 폴더를 이동
for filename in os.listdir(source_path):
    file_path = os.path.join(source_path, filename)
    if os.path.isfile(file_path) or os.path.isdir(file_path):
        shutil.move(file_path, destination_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
